# 01. Data Annotation Script

Jupyter notebook to script auto annotation.

In [ ]:
import os
import shutil
import random
from pathlib import Path
from ultralytics import YOLO

# ==========================================
# PATH CONFIGURATION 
# ==========================================
RAW_DIR = Path("../data/raw/Fruit And Vegetable Diseases Dataset")
PROCESSED_DIR = Path("../data/processed")

# Data split ratios
TRAIN_RATIO = 0.80  # 80% Training
VAL_RATIO = 0.15    # 15% Validation during training
TEST_RATIO = 0.05   # 5% Independent testing

def setup_directories():
    """Create the standard 3-folder structure for YOLO (train, val, test)."""
    for split in ['train', 'val', 'test']:
        (PROCESSED_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (PROCESSED_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

def create_yaml(classes):
    """Auto-generate dataset.yaml including the test split path."""
    yaml_path = PROCESSED_DIR / "dataset.yaml"
    with open(yaml_path, 'w', encoding='utf-8') as f:
        f.write(f"path: {PROCESSED_DIR.resolve().as_posix()}\n")
        f.write("train: images/train\n")
        f.write("val: images/val\n")
        f.write("test: images/test\n\n")  # Include test split path
        f.write("names:\n")
        for idx, cls_name in enumerate(classes):
            f.write(f"  {idx}: {cls_name}\n")
    print(f"✅ Config file created: {yaml_path}")

def process_data():
    setup_directories()
    
    print("⏳ Loading YOLOv8n for auto-annotation...")
    auto_model = YOLO("yolov8n.pt") 
    
    classes = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
    create_yaml(classes)
    
    total_images = 0
    missing_boxes = 0

    for class_id, class_name in enumerate(classes):
        class_dir = RAW_DIR / class_name
        images = [p for p in class_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
        
        if not images:
            continue
            
        print(f"🔄 Processing class [{class_id}] {class_name}: {len(images)} images...")
        random.shuffle(images)
        
        # Split data into 3 sets
        val_idx = int(len(images) * TRAIN_RATIO)
        test_idx = val_idx + int(len(images) * VAL_RATIO)
        
        train_imgs = images[:val_idx]
        val_imgs = images[val_idx:test_idx]
        test_imgs = images[test_idx:]
        
        # Iterate over all 3 splits
        splits = [
            (train_imgs, 'train'),
            (val_imgs, 'val'),
            (test_imgs, 'test')
        ]
        
        for img_list, split_name in splits:
            for img_path in img_list:
                results = auto_model(img_path, verbose=False)
                boxes = results[0].boxes.xywhn 
                
                new_img_path = PROCESSED_DIR / 'images' / split_name / f"{class_name}_{img_path.name}"
                shutil.copy(img_path, new_img_path)
                
                label_path = PROCESSED_DIR / 'labels' / split_name / f"{class_name}_{img_path.stem}.txt"
                with open(label_path, 'w') as f:
                    if len(boxes) > 0:
                        for box in boxes.tolist():
                            f.write(f"{class_id} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n")
                    else:
                        # Fallback when no object detected
                        box = [0.5, 0.5, 0.9, 0.9] 
                        missing_boxes += 1
                        f.write(f"{class_id} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n")
                
                total_images += 1

    print("\n" + "="*50)
    print(f"🎉 DATA PROCESSING COMPLETE!")
    print(f"Total images processed: {total_images}")
    print(f"Images using fallback box: {missing_boxes}")
    print("="*50)

if __name__ == "__main__":
    process_data()